# 02 — Preprocessing & EDA

_Placeholder — Cleaning (drop inf/NaN, constant + duplicate columns), remove leakage features, Min-Max scaling, label encoding, and the stratified 80/20 split. Built together after the data inventory is confirmed._

In [1]:
import glob, os, hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import joblib

DATA_DIR = r"D:\Capstone Project\MachineLearningCVE"
PROCESSED_DIR = os.path.join(os.getcwd(), "..", "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "..", "models")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
frames = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = df.columns.str.strip()
    frames.append(df)
data = pd.concat(frames, ignore_index=True)
print("Loaded:", data.shape)

Loaded: (2830743, 79)


In [2]:
# Any label starting with "Web Attack" -> single "Web Attack" class
data["Label"] = data["Label"].str.strip()
mask = data["Label"].str.startswith("Web Attack")
data.loc[mask, "Label"] = "Web Attack"

# Drop the two ultra-rare classes: too few samples to train or evaluate reliably.
data = data[~data["Label"].isin(["Heartbleed", "Infiltration"])].reset_index(drop=True)

print("Classes after merge + drop:", data["Label"].nunique())
print(data["Label"].value_counts())

Classes after merge + drop: 11
Label
BENIGN              2273097
DoS Hulk             231073
PortScan             158930
DDoS                 128027
DoS GoldenEye         10293
FTP-Patator            7938
SSH-Patator            5897
DoS slowloris          5796
DoS Slowhttptest       5499
Web Attack             2180
Bot                    1966
Name: count, dtype: int64


In [3]:
before = len(data)
data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(f"Rows removed (inf/NaN): {before - len(data)}")
print("Shape now:", data.shape)

Rows removed (inf/NaN): 2867
Shape now: (2827829, 79)


In [4]:
features = data.drop(columns=["Label"])

# constant (zero-variance) columns
nun = features.nunique(dropna=False)
constant_cols = nun[nun <= 1].index.tolist()

# duplicate columns (e.g. 'Fwd Header Length.1')
seen, duplicate_cols = {}, []
for col in features.columns:
    h = hashlib.md5(pd.util.hash_pandas_object(features[col], index=False).values.tobytes()).hexdigest()
    if h in seen and features[col].equals(features[seen[h]]):
        duplicate_cols.append(col)
    else:
        seen.setdefault(h, col)

print("Constant columns:", constant_cols)
print("Duplicate columns:", duplicate_cols)

data = data.drop(columns=constant_cols + duplicate_cols)
print("Shape after column drops:", data.shape)

Constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Duplicate columns: ['Bwd URG Flags', 'SYN Flag Count', 'CWE Flag Count', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Bwd Packets']
Shape after column drops: (2827829, 66)


In [5]:
# In the MachineLearningCVE version the only such column present is usually
# 'Destination Port'. We intersect with what's actually here, so this is safe.
leakage_candidates = ["Flow ID", "Source IP", "Source Port",
                      "Destination IP", "Destination Port", "Timestamp"]
leakage_present = [c for c in leakage_candidates if c in data.columns]
data = data.drop(columns=leakage_present)
print("Dropped leakage features:", leakage_present)
print("Shape now:", data.shape)

Dropped leakage features: ['Destination Port']
Shape now: (2827829, 65)


In [6]:
X = data.drop(columns=["Label"])
y = data["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Feature count:", X_train.shape[1])
# confirm stratification preserved the tiny classes in both splits
print("\nTrain class counts:\n", y_train.value_counts())
print("\nTest class counts:\n", y_test.value_counts())

Train: (2262263, 64)  Test: (565566, 64)
Feature count: 64

Train class counts:
 Label
BENIGN              1817056
DoS Hulk             184099
PortScan             127043
DDoS                 102420
DoS GoldenEye          8234
FTP-Patator            6348
SSH-Patator            4718
DoS slowloris          4637
DoS Slowhttptest       4399
Web Attack             1744
Bot                    1565
Name: count, dtype: int64

Test class counts:
 Label
BENIGN              454264
DoS Hulk             46025
PortScan             31761
DDoS                 25605
DoS GoldenEye         2059
FTP-Patator           1587
SSH-Patator           1179
DoS slowloris         1159
DoS Slowhttptest      1100
Web Attack             436
Bot                    391
Name: count, dtype: int64


In [7]:
# Fit the scaler ONLY on training data, then apply to both.
# Fitting on the full set would leak test-set range into training —
# exactly the kind of leakage this project is built to avoid.
scaler = MinMaxScaler()
feature_names = X_train.columns.tolist()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)
print("Scaled. X_train dtype:", X_train.dtype, " range:", X_train.min(), "to", X_train.max())

Scaled. X_train dtype: float32  range: 0.0 to 1.0


In [8]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print("Label mapping:")
for i, c in enumerate(le.classes_):
    print(f"  {i:2d} -> {c}")

Label mapping:
   0 -> BENIGN
   1 -> Bot
   2 -> DDoS
   3 -> DoS GoldenEye
   4 -> DoS Hulk
   5 -> DoS Slowhttptest
   6 -> DoS slowloris
   7 -> FTP-Patator
   8 -> PortScan
   9 -> SSH-Patator
  10 -> Web Attack


In [9]:
joblib.dump(
    {"X_train": X_train, "X_test": X_test,
     "y_train": y_train_enc, "y_test": y_test_enc,
     "feature_names": feature_names,
     "label_classes": le.classes_.tolist()},
    os.path.join(PROCESSED_DIR, "preprocessed.joblib"), compress=3
)
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
joblib.dump(le, os.path.join(MODELS_DIR, "label_encoder.joblib"))
print("Saved preprocessed data and encoders.")

Saved preprocessed data and encoders.


In [10]:
import joblib, os
print("N features:", len(feature_names))
print(feature_names)

N features: 64
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd